# MINE4201 - SR - Taller 1 - Experimentación y optimización para K vecinos más cercanos

**Grupo 3**

Lina María Ojeda Amaya Código: 202112324

Diego Felipe Tolosa Código: 201413235

Juan Manuel Rivera López Código: 201534131

Johana Alejandra Rátiva Mora Código: 202513844

# Importación de librerías y datos

In [1]:
import numpy as np
import os
import pandas as pd
import random
from matplotlib import pyplot as plt
import time

from scipy.sparse import csr_matrix

from sklearn.preprocessing import normalize
from tqdm import tqdm

#Para garantizar reproducibilidad en resultados
seed = 10
random.seed(seed)
np.random.seed(seed)

In [2]:
base_path = "data/"
links_path = os.path.join(base_path, "link.csv")
movies_path = os.path.join(base_path, "movie.csv")
ratings_path = os.path.join(base_path, "rating.csv")

In [3]:
ratings = pd.read_csv(ratings_path)
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [4]:
movies = pd.read_csv(movies_path)
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
test = ratings.groupby("userId", group_keys=False).sample(frac=0.2, random_state=42)
train = ratings.drop(test.index)

In [6]:
user_cat = train.userId.astype("category")
item_cat = train.movieId.astype("category")

user_ids = user_cat.cat.codes
item_ids = item_cat.cat.codes

R_train = csr_matrix(
    (train.rating, (user_ids, item_ids))
)

In [7]:
user_map = dict(zip(user_cat.cat.categories, range(len(user_cat.cat.categories))))
item_map = dict(zip(item_cat.cat.categories, range(len(item_cat.cat.categories))))

# Evaluación de similitud usuario-usuario

In [8]:
N = R_train.shape[0]

counts = np.diff(R_train.indptr)
sums = np.add.reduceat(R_train.data, R_train.indptr[:-1])
means = sums / counts

### Función de cálculo de rating - User-User

Se crea una función capaz de implementar la predicción de ratings mediante **colaboración basada en usuarios (user-user collaborative filtering)**. Esta función calcula la predicción del rating de un usuario para un ítem específico utilizando los ratings de usuarios similares.

#### `predict_rating_user()`

La función toma como entrada los parámetros necesarios para realizar la predicción:

- **R**: Matriz dispersa (usuario × ítem) que contiene los ratings de entrenamiento
- **userId**: Índice del usuario para el cual se realiza la predicción
- **movieId**: Índice del ítem a predecir
- **neighbours**: Array con los índices de los usuarios más similares
- **sims**: Array con las similitudes correspondientes a cada vecino
- **K**: Número máximo de vecinos considerados
- **sim_threshold**: Valor mínimo de similitud para incluir un vecino en la predicción
- **means**: Promedio de ratings para cada usuario
- **weighting**: Booleano para aplicar ponderación de McLaughlin (opcional)
- **B**: Matriz de co-rated (elementos en común precomputados) (requerida si weighting=True)
- **beta**: Parámetro de escala para la ponderación de McLaughlin (por defecto 50)

#### Proceso de cálculo

1. **Extracción de ratings**: Se obtienen los ratings que los usuarios similares han dado al ítem objetivo
2. **Filtrado**: Se descartan vecinos que no calificaron el ítem o cuya similitud es inferior al threshold
3. **Centrado**: Se restan los promedios de cada usuario para normalizar las escalas
4. **Ponderación (opcional)**: Se aplica el método de McLaughlin para penalizar similitudes basadas en pocos datos comunes
5. **Combinación ponderada**: Se calcula la predicción como el promedio del usuario más una desviación ponderada

#### Fórmula matemática

$$\hat{r}_{u,i} = \bar{r}_u + \frac{\sum_{v \in N_i(u)} w_v \cdot \text{sim}(u,v) \cdot (r_{v,i} - \bar{r}_v)}{\sum_{v \in N_i(u)} |w_v \cdot \text{sim}(u,v)|}$$

Donde $w_v = \min(\text{corated}(u,v) / \beta, 1.0)$ cuando se aplica ponderación de McLaughlin.

Esta implementación permite optimizar las predicciones al penalizar similitudes con poca base empírica, mejorando la robustez del modelo.

In [9]:
def predict_rating_user(
        R,
        userId,
        movieId,
        neighbours,
        sims,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    # --- get ratings for the movie ---
    ratings = R[neighbours, movieId].toarray().flatten()

    # keep neighbours that rated the movie
    mask = (ratings > 0) & (sims >= sim_threshold)

    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # --- mean centered ratings ---
    centered = ratings - means[neighbours]

    # --- McLaughlin significance weighting ---
    if weighting:

        intersections = B[userId]

        weights = np.minimum(intersections / beta, 1.0)

        weights = weights[mask]

        sims *= weights

    numerator = np.sum(sims * centered)
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return np.nan

    pred = means[userId] + numerator / denominator

    return pred

### Función de evaluación RMSE - User-User

Esta función calcula el error cuadrático medio (RMSE) de las predicciones generadas por el modelo de filtrado colaborativo basado en usuarios. Permite comparar el desempeño del sistema bajo diferentes configuraciones de vecinos, similitud y ponderación.

#### `evaluate_rmse_user()`

**Parámetros:**

- **R**: Matriz dispersa de ratings de entrenamiento

- **test_df**: DataFrame de prueba con índices de usuario e ítem

- **neighbors_list**: Matriz de vecinos para cada usuario

- **sims_list**: Matriz de similitudes para cada usuario

- **K**: Número de vecinos considerados

- **sim_threshold**: Umbral mínimo de similitud

- **means**: Promedio de ratings por usuario

- **weighting**: Si aplica ponderación de McLaughlin

- **B**: Matriz de co-rated (elementos en común precomputados)

- **beta**: Parámetro de escala para ponderación


#### Proceso de cálculo

1. Para cada usuario y película en el set de prueba, predice el rating usando la función de predicción.
2. Guarda la predicción y el rating real.
3. Calcula el RMSE entre las predicciones y los ratings reales.

#### Fórmula matemática

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (\hat{r}_i - r_i)^2}$$

Donde $\hat{r}_i$ es la predicción y $r_i$ el rating real.

Esta función permite evaluar la calidad de las predicciones y comparar configuraciones del modelo.

In [10]:
def evaluate_rmse_user(
        R,
        test_df,
        neighbors_list,
        sims_list,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    preds = []
    truths = []
    missing = 0

    users = test_df.user_idx.values
    movies = test_df.movie_idx.values
    ratings = test_df.rating.values

    for i in range(len(users)):

        user_idx = users[i]
        movie_idx = movies[i]
        rating = ratings[i]

        pred = predict_rating_user(
            R,
            user_idx,
            movie_idx,
            neighbors_list[user_idx],
            sims_list[user_idx],
            K,
            sim_threshold,
            means,
            weighting,
            B,
            beta=beta
        )

        if not np.isnan(pred):
            preds.append(pred)
            truths.append(rating)
        else:
            missing+=1

    preds = np.array(preds)
    truths = np.array(truths)

    rmse = np.sqrt(np.mean((preds - truths) ** 2))

    return rmse, missing

## Sub muestra de test (n=5.000)

In [11]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.1,0.2], desc="sim_threshold", leave=False):
            weight=False
            rmse,missing = evaluate_rmse_user(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                means,
                weight,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse, missing))
            print((metric, K, sim_threshold, weight, rmse, missing))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.05408
max sim: 0.9224


('cosine', 25, 0, False, np.float64(0.8754917505297362), 503)


('cosine', 25, 0.1, False, np.float64(0.8754917505297362), 503)


('cosine', 25, 0.2, False, np.float64(0.8722073531995769), 521)


...

('jaccard', 50, 0.2, False, np.float64(0.868761743545282), 1405)


('jaccard', 100, 0, False, np.float64(0.8537566672461587), 235)


('jaccard', 100, 0.1, False, np.float64(0.8524950191137718), 281)




Similarity metric: 100%|██████████| 3/3 [00:18<00:00,  6.27s/it]

('jaccard', 100, 0.2, False, np.float64(0.8523834179918015), 1349)


In [12]:
results

[('cosine', 25, 0, False, np.float64(0.8754917505297362), 503),
 ('cosine', 25, 0.1, False, np.float64(0.8754917505297362), 503),
 ('cosine', 25, 0.2, False, np.float64(0.8722073531995769), 521),
 ('cosine', 50, 0, False, np.float64(0.8523012503228641), 310),
 ('cosine', 50, 0.1, False, np.float64(0.8523012503228641), 310),
 ('cosine', 50, 0.2, False, np.float64(0.8502698939943181), 333),
 ('cosine', 100, 0, False, np.float64(0.8449797573858153), 180),
 ('cosine', 100, 0.1, False, np.float64(0.8447839577673898), 180),
 ('cosine', 100, 0.2, False, np.float64(0.8426032885984476), 217),
 ('pearson', 25, 0, False, np.float64(0.8691050174067901), 1190),
 ('pearson', 25, 0.1, False, np.float64(0.8689913873985596), 1194),
 ('pearson', 25, 0.2, False, np.float64(0.8583866700371544), 2441),
 ('pearson', 50, 0, False, np.float64(0.8562168766214626), 764),
 ('pearson', 50, 0.1, False, np.float64(0.8558558407804776), 775),
 ('pearson', 50, 0.2, False, np.float64(0.858867165442867), 2289),
 ('pears

Viendo que el coeficiente de Jaccard tiene los peores resultados en la exploración incial, se excluirá en iteraciones futuras.

## Submuestra de test (n=250.000)

In [13]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(250_000, random_state=42)

for metric in tqdm(["cosine", "pearson"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.1,0.2], desc="sim_threshold", leave=False):

            rmse,missing = evaluate_rmse_user(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                means,
                weighting=False,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse, missing))
            print((metric, K, sim_threshold, weight, rmse, missing))

Similarity metric:   0%|          | 0/2 [00:00<?, ?it/s]

min sim: 0.05408
max sim: 0.9224


('cosine', 25, 0, False, np.float64(0.8806656033893725), 26556)


('cosine', 25, 0.1, False, np.float64(0.8806656816262836), 26556)


('cosine', 25, 0.2, False, np.float64(0.8800858731606602), 27586)


...

('pearson', 50, 0.2, False, np.float64(0.8686067719377053), 114607)


('pearson', 100, 0, False, np.float64(0.8581761219436683), 23963)


('pearson', 100, 0.1, False, np.float64(0.8577758757408119), 24882)




Similarity metric: 100%|██████████| 2/2 [08:01<00:00, 240.88s/it]

('pearson', 100, 0.2, False, np.float64(0.8613557348177775), 110667)


In [14]:
results

[('cosine', 25, 0, False, np.float64(0.8806656033893725), 26556),
 ('cosine', 25, 0.1, False, np.float64(0.8806656816262836), 26556),
 ('cosine', 25, 0.2, False, np.float64(0.8800858731606602), 27586),
 ('cosine', 50, 0, False, np.float64(0.8626560533342876), 16337),
 ('cosine', 50, 0.1, False, np.float64(0.8626249556059443), 16342),
 ('cosine', 50, 0.2, False, np.float64(0.8616414334893181), 17932),
 ('cosine', 100, 0, False, np.float64(0.8511609839977688), 9590),
 ('cosine', 100, 0.1, False, np.float64(0.8511634239168042), 9599),
 ('cosine', 100, 0.2, False, np.float64(0.8501761949048814), 11753),
 ('pearson', 25, 0, False, np.float64(0.8892202755242916), 61219),
 ('pearson', 25, 0.1, False, np.float64(0.8890144702789837), 61569),
 ('pearson', 25, 0.2, False, np.float64(0.877927841901627), 122165),
 ('pearson', 50, 0, False, np.float64(0.8733750963456223), 39721),
 ('pearson', 50, 0.1, False, np.float64(0.8730812963952731), 40308),
 ('pearson', 50, 0.2, False, np.float64(0.8686067719

## Implementación de Ponderación de McLaughlin

Se calculará McNally para el mejor tipo de similitud en el modelo usuario-usuario, que en este caso es Pearson, para esta se iterará para 3 Valores de k vecinos, 3 valores de umbral de similitud, y 5 posibles valores de ponderación de McLaughlin (incluyendo NO implementarla)

## Submuestra de test (n=5.000)

In [15]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(5_000, random_state=42)

for metric in tqdm(["pearson"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/users_{metric}_top100_co_rated.npz") as data:
        top_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.1, 0.2], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse,missing = evaluate_rmse_user(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse, missing))
                    print((metric, K, sim_threshold, weight, None, rmse, missing))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse, missing = evaluate_rmse_user(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse, missing))
                        print((metric, K, sim_threshold, weight, beta, rmse, missing))

Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: -1.0
max sim: 0.9672295
min co_rated: 0
max co_rated: 3642


('pearson', 25, 0, False, None, np.float64(0.8691050174067901), 1190)
('pearson', 25, 0, True, 5, np.float64(0.869541483445122), 1190)
('pearson', 25, 0, True, 10, np.float64(0.8702049356821959), 1190)
('pearson', 25, 0, True, 25, np.float64(0.87163840420329), 1190)
('pearson', 25, 0, True, 50, np.float64(0.8734886457752691), 1190)


('pearson', 25, 0, True, 100, np.float64(0.8740969481708274), 1190)
('pearson', 25, 0.1, False, None, np.float64(0.8689913873985596), 1194)
('pearson', 25, 0.1, True, 5, np.float64(0.8694283691391966), 1194)
('pearson', 25, 0.1, True, 10, np.float64(0.8700926047211486), 1194)
('pearson', 25, 0.1, True, 25, np.float64(0.8715277634910148), 1194)
('pearson', 25, 0.1, True, 50, np.float64(0.8733801821903403), 1194)


('pearson', 25, 0.1, True, 100, np.float64(0.873989138665914), 1194)
('pearson', 25, 0.2, False, None, np.float64(0.8583866700371544), 2441)
('pearson', 25, 0.2, True, 5, np.float64(0.8589615137015859), 2441)
('pearson', 25, 0.2, True, 10, np.float64(0.8596208752752366), 2441)
('pearson', 25, 0.2, True, 25, np.float64(0.8606621058303862), 2441)
('pearson', 25, 0.2, True, 50, np.float64(0.8623665869400406), 2441)


...

('pearson', 100, 0, False, None, np.float64(0.845019019260748), 463)
('pearson', 100, 0, True, 5, np.float64(0.8454372405738101), 463)
('pearson', 100, 0, True, 10, np.float64(0.8461970071268775), 463)
('pearson', 100, 0, True, 25, np.float64(0.845326369799388), 463)
('pearson', 100, 0, True, 50, np.float64(0.8453878527769222), 463)


('pearson', 100, 0, True, 100, np.float64(0.8456585230023135), 463)
('pearson', 100, 0.1, False, None, np.float64(0.8438555373362965), 479)
('pearson', 100, 0.1, True, 5, np.float64(0.8442816617654119), 479)
('pearson', 100, 0.1, True, 10, np.float64(0.8450696049932551), 479)
('pearson', 100, 0.1, True, 25, np.float64(0.8442205023030612), 479)
('pearson', 100, 0.1, True, 50, np.float64(0.8442092956798561), 479)


('pearson', 100, 0.1, True, 100, np.float64(0.8444469783068906), 479)
('pearson', 100, 0.2, False, None, np.float64(0.8589023825147732), 2206)
('pearson', 100, 0.2, True, 5, np.float64(0.860551163884366), 2206)
('pearson', 100, 0.2, True, 10, np.float64(0.8613491524527092), 2206)
('pearson', 100, 0.2, True, 25, np.float64(0.8604780333313509), 2206)
('pearson', 100, 0.2, True, 50, np.float64(0.8614444344358139), 2206)




Similarity metric: 100%|██████████| 1/1 [00:26<00:00, 26.39s/it]

('pearson', 100, 0.2, True, 100, np.float64(0.8612662001501917), 2206)


In [16]:
results

[('pearson', 25, 0, False, None, np.float64(0.8691050174067901), 1190),
 ('pearson', 25, 0, True, 5, np.float64(0.869541483445122), 1190),
 ('pearson', 25, 0, True, 10, np.float64(0.8702049356821959), 1190),
 ('pearson', 25, 0, True, 25, np.float64(0.87163840420329), 1190),
 ('pearson', 25, 0, True, 50, np.float64(0.8734886457752691), 1190),
 ('pearson', 25, 0, True, 100, np.float64(0.8740969481708274), 1190),
 ('pearson', 25, 0.1, False, None, np.float64(0.8689913873985596), 1194),
 ('pearson', 25, 0.1, True, 5, np.float64(0.8694283691391966), 1194),
 ('pearson', 25, 0.1, True, 10, np.float64(0.8700926047211486), 1194),
 ('pearson', 25, 0.1, True, 25, np.float64(0.8715277634910148), 1194),
 ('pearson', 25, 0.1, True, 50, np.float64(0.8733801821903403), 1194),
 ('pearson', 25, 0.1, True, 100, np.float64(0.873989138665914), 1194),
 ('pearson', 25, 0.2, False, None, np.float64(0.8583866700371544), 2441),
 ('pearson', 25, 0.2, True, 5, np.float64(0.8589615137015859), 2441),
 ('pearson', 2

## Submuestra de test (n=250.000)

In [17]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(250_000, random_state=42)

for metric in tqdm(["pearson"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/users_{metric}_top100_co_rated.npz") as data:
        top_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.1, 0.2], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse,missing = evaluate_rmse_user(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse, missing))
                    print((metric, K, sim_threshold, weight, None, rmse, missing))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse, missing = evaluate_rmse_user(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse, missing))
                        print((metric, K, sim_threshold, weight, beta, rmse, missing))

Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: -1.0
max sim: 0.9672295
min co_rated: 0
max co_rated: 3642


('pearson', 25, 0, False, None, np.float64(0.8892202755242916), 61219)
('pearson', 25, 0, True, 5, np.float64(0.8894383992489105), 61219)
('pearson', 25, 0, True, 10, np.float64(0.8897755943753323), 61219)
('pearson', 25, 0, True, 25, np.float64(0.8904079461075728), 61219)
('pearson', 25, 0, True, 50, np.float64(0.8910818096581642), 61219)


('pearson', 25, 0, True, 100, np.float64(0.8919589438108231), 61219)
('pearson', 25, 0.1, False, None, np.float64(0.8890144702789837), 61569)
('pearson', 25, 0.1, True, 5, np.float64(0.88923256560022), 61569)
('pearson', 25, 0.1, True, 10, np.float64(0.8895723220257263), 61569)
('pearson', 25, 0.1, True, 25, np.float64(0.8902076922746149), 61569)
('pearson', 25, 0.1, True, 50, np.float64(0.8908836676893389), 61569)


('pearson', 25, 0.1, True, 100, np.float64(0.8917542223553748), 61569)
('pearson', 25, 0.2, False, None, np.float64(0.877927841901627), 122165)
('pearson', 25, 0.2, True, 5, np.float64(0.8782093552925272), 122165)
('pearson', 25, 0.2, True, 10, np.float64(0.8785051295657981), 122165)
('pearson', 25, 0.2, True, 25, np.float64(0.878913973288406), 122165)
('pearson', 25, 0.2, True, 50, np.float64(0.8792564135426343), 122165)


...

('pearson', 100, 0, False, None, np.float64(0.8581761219436683), 23963)
('pearson', 100, 0, True, 5, np.float64(0.8583942314060745), 23963)
('pearson', 100, 0, True, 10, np.float64(0.8585982513536641), 23963)
('pearson', 100, 0, True, 25, np.float64(0.8589647409506121), 23963)
('pearson', 100, 0, True, 50, np.float64(0.8595523837997658), 23963)


('pearson', 100, 0, True, 100, np.float64(0.8602099614333594), 23963)
('pearson', 100, 0.1, False, None, np.float64(0.8577758757408119), 24882)
('pearson', 100, 0.1, True, 5, np.float64(0.8580081032281301), 24882)
('pearson', 100, 0.1, True, 10, np.float64(0.8582257541087062), 24882)
('pearson', 100, 0.1, True, 25, np.float64(0.8586045341034575), 24882)
('pearson', 100, 0.1, True, 50, np.float64(0.8591928653777087), 24882)


('pearson', 100, 0.1, True, 100, np.float64(0.8598558910152968), 24882)
('pearson', 100, 0.2, False, None, np.float64(0.8613557348177775), 110667)
('pearson', 100, 0.2, True, 5, np.float64(0.8615532524398327), 110667)
('pearson', 100, 0.2, True, 10, np.float64(0.8618340698150266), 110667)
('pearson', 100, 0.2, True, 25, np.float64(0.8620712305427843), 110667)
('pearson', 100, 0.2, True, 50, np.float64(0.8623011527052888), 110667)




Similarity metric: 100%|██████████| 1/1 [21:36<00:00, 1296.72s/it]

('pearson', 100, 0.2, True, 100, np.float64(0.8626625903971419), 110667)


In [18]:
results

[('pearson', 25, 0, False, None, np.float64(0.8892202755242916), 61219),
 ('pearson', 25, 0, True, 5, np.float64(0.8894383992489105), 61219),
 ('pearson', 25, 0, True, 10, np.float64(0.8897755943753323), 61219),
 ('pearson', 25, 0, True, 25, np.float64(0.8904079461075728), 61219),
 ('pearson', 25, 0, True, 50, np.float64(0.8910818096581642), 61219),
 ('pearson', 25, 0, True, 100, np.float64(0.8919589438108231), 61219),
 ('pearson', 25, 0.1, False, None, np.float64(0.8890144702789837), 61569),
 ('pearson', 25, 0.1, True, 5, np.float64(0.88923256560022), 61569),
 ('pearson', 25, 0.1, True, 10, np.float64(0.8895723220257263), 61569),
 ('pearson', 25, 0.1, True, 25, np.float64(0.8902076922746149), 61569),
 ('pearson', 25, 0.1, True, 50, np.float64(0.8908836676893389), 61569),
 ('pearson', 25, 0.1, True, 100, np.float64(0.8917542223553748), 61569),
 ('pearson', 25, 0.2, False, None, np.float64(0.877927841901627), 122165),
 ('pearson', 25, 0.2, True, 5, np.float64(0.8782093552925272), 122165

# Evaluación de similitud item-item

In [19]:
R_items = R_train.T

N = R_items.shape[0]

counts = np.diff(R_items.indptr)
sums = np.add.reduceat(R_items.data, R_items.indptr[:-1])
item_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts!=0)

### Función de cálculo de rating - Item-Item

Se crea una función capaz de implementar la predicción de ratings mediante **colaboración basada en ítems (item-item collaborative filtering)**. Esta función calcula la predicción del rating de un usuario para un ítem específico utilizando los ratings del usuario para ítems similares.

#### `predict_rating_item()`

La función toma como entrada los parámetros necesarios para realizar la predicción:

- **R**: Matriz dispersa (usuario × ítem) que contiene los ratings de entrenamiento
- **userId**: Índice del usuario para el cual se realiza la predicción
- **movieId**: Índice del ítem a predecir
- **neighbours**: Array con los índices de los ítems más similares
- **sims**: Array con las similitudes correspondientes a cada ítem vecino
- **K**: Número máximo de vecinos considerados
- **sim_threshold**: Valor mínimo de similitud para incluir un vecino en la predicción
- **means**: Promedio de ratings para cada ítem
- **weighting**: Booleano para aplicar ponderación de McLaughlin (opcional)
- **B**: Matriz de co-rated (elementos en común precomputados) (requerida si weighting=True)
- **beta**: Parámetro de escala para la ponderación de McLaughlin (por defecto 50)

#### Proceso de cálculo

1. **Extracción de ratings**: Se obtienen los ratings que el usuario ha dado a los ítems similares
2. **Filtrado**: Se descartan ítems que el usuario no calificó o cuya similitud es inferior al threshold
3. **Centrado**: Se restan los promedios de cada ítem vecino para normalizar las escalas
4. **Ponderación (opcional)**: Se aplica el método de McLaughlin para penalizar similitudes basadas en pocos datos comunes
5. **Combinación ponderada**: Se calcula la predicción como el promedio del ítem más una desviación ponderada

#### Fórmula matemática

$$\hat{r}_{u,i} = \bar{r}_i + \frac{\sum_{j \in N(i)} w_j \cdot \text{sim}(i,j) \cdot (r_{u,j} - \bar{r}_j)}{\sum_{j \in N(i)} |w_j \cdot \text{sim}(i,j)|}$$

Donde $w_j = \min(\text{corated}(i,j) / \beta, 1.0)$ cuando se aplica ponderación de McLaughlin.

Esta implementación permite hacer predicciones basadas en la similitud entre ítems, capturando patrones de preferencia a través de ítems relacionados.

In [20]:
def predict_rating_item(
        R,
        userId,
        movieId,
        neighbours,
        sims,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    # --- get ratings for the movie ---
    ratings = R[userId, neighbours].toarray().flatten()

    # keep neighbours that rated the movie
    mask = (ratings > 0) & (sims >= sim_threshold)

    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # --- mean centered ratings ---
    centered = ratings - means[neighbours]

    # --- McLaughlin significance weighting ---
    if weighting:

        intersections = B[movieId]

        weights = np.minimum(intersections / beta, 1.0)

        weights = weights[mask]

        sims *= weights

    numerator = np.sum(sims * centered)
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return np.nan

    pred = means[movieId] + numerator / denominator

    return pred


### Función de evaluación RMSE - Item-Item

Esta función calcula el error cuadrático medio (RMSE) de las predicciones generadas por el modelo de filtrado colaborativo basado en ítems. Permite comparar el desempeño del sistema bajo diferentes configuraciones de vecinos, similitud y ponderación.

#### `evaluate_rmse_item()`

**Parámetros:**

- **R**: Matriz dispersa de ratings de entrenamiento

- **test_df**: DataFrame de prueba con índices de usuario e ítem

- **neighbors_list**: Matriz de vecinos para cada ítem

- **sims_list**: Matriz de similitudes para cada ítem

- **K**: Número de vecinos considerados

- **sim_threshold**: Umbral mínimo de similitud

- **means**: Promedio de ratings por ítem

- **weighting**: Si aplica ponderación de McLaughlin

- **B**: Matriz de co-rated (elementos en común precomputados)

- **beta**: Parámetro de escala para ponderación


#### Proceso de cálculo

1. Para cada usuario e ítem en el set de prueba, predice el rating usando la función de predicción item-item.
2. Guarda la predicción y el rating real.
3. Calcula el RMSE entre las predicciones y los ratings reales.

#### Fórmula matemática

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (\hat{r}_i - r_i)^2}$$

Donde $\hat{r}_i$ es la predicción y $r_i$ el rating real.

Esta función permite evaluar la calidad de las predicciones basadas en similitud de ítems y comparar configuraciones del modelo.

In [21]:
def evaluate_rmse_item(
        R,
        test_df,
        neighbors_list,
        sims_list,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50):

    preds = []
    truths = []
    missing = 0

    users = test_df.user_idx.values
    movies = test_df.movie_idx.values
    ratings = test_df.rating.values

    for i in range(len(users)):

        user_idx = users[i]
        movie_idx = movies[i]
        rating = ratings[i]

        pred = predict_rating_item(
            R,
            user_idx,
            movie_idx,
            neighbors_list[movie_idx],
            sims_list[movie_idx],
            K,
            sim_threshold,
            means,
            weighting,
            B,
            beta=beta
        )

        if not np.isnan(pred):
            preds.append(pred)
            truths.append(rating)
        else:
            missing+=1

    preds = np.array(preds)
    truths = np.array(truths)

    rmse = np.sqrt(np.mean((preds - truths) ** 2))

    return rmse, missing

## Submuestra de test (n=5.000)

In [22]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

In [23]:
for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    print(f"Dimensiones neighbours: {top_k_indices.shape}")
    print(f"Número de items en R_items: {R_items.shape[0]}")
    print(f"Longitud de item_means: {len(item_means)}")

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.1,0.2], desc="sim_threshold", leave=False):
            weight=False
            rmse,missing = evaluate_rmse_item(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                item_means,
                weighting=False,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse, missing))
            print((metric, K, sim_threshold, weight, rmse, missing))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0
Dimensiones neighbours: (25865, 100)
Número de items en R_items: 25865
Longitud de item_means: 138493


('cosine', 25, 0, False, np.float64(1.0056557408107982), 269)


('cosine', 25, 0.1, False, np.float64(1.0040764327788705), 282)


('cosine', 25, 0.2, False, np.float64(1.0174789935620738), 693)


...

('jaccard', 50, 0.2, False, np.float64(1.0166463433275197), 2450)


('jaccard', 100, 0, False, np.float64(0.9695694441791524), 100)


('jaccard', 100, 0.1, False, np.float64(0.9809795511355256), 528)




Similarity metric: 100%|██████████| 3/3 [00:10<00:00,  3.60s/it]

('jaccard', 100, 0.2, False, np.float64(1.017861235031958), 2449)


In [24]:
results

[('cosine', 25, 0, False, np.float64(1.0056557408107982), 269),
 ('cosine', 25, 0.1, False, np.float64(1.0040764327788705), 282),
 ('cosine', 25, 0.2, False, np.float64(1.0174789935620738), 693),
 ('cosine', 50, 0, False, np.float64(0.9839930175791636), 141),
 ('cosine', 50, 0.1, False, np.float64(0.982269411650195), 159),
 ('cosine', 50, 0.2, False, np.float64(0.9980906621330419), 626),
 ('cosine', 100, 0, False, np.float64(0.9819672028860388), 76),
 ('cosine', 100, 0.1, False, np.float64(0.9825753650531198), 96),
 ('cosine', 100, 0.2, False, np.float64(0.9996290935273029), 599),
 ('pearson', 25, 0, False, np.float64(1.0053548899690898), 269),
 ('pearson', 25, 0.1, False, np.float64(1.0037752517731051), 282),
 ('pearson', 25, 0.2, False, np.float64(1.0170274422324193), 693),
 ('pearson', 50, 0, False, np.float64(0.9839066705010061), 141),
 ('pearson', 50, 0.1, False, np.float64(0.9821359150469063), 159),
 ('pearson', 50, 0.2, False, np.float64(0.9980266217232427), 626),
 ('pearson', 1

## Submuestra de test (n=250.000)

In [25]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(250_000, random_state=42)

In [26]:
for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.1,0.2], desc="sim_threshold", leave=False):
            weight=False
            rmse,missing = evaluate_rmse_item(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                means,
                weighting=False,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse, missing))
            print((metric, K, sim_threshold, weight, rmse, missing))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0


('cosine', 25, 0, False, np.float64(1.0031778877693514), 13961)


('cosine', 25, 0.1, False, np.float64(1.0029982138508473), 14284)


('cosine', 25, 0.2, False, np.float64(1.0032462710724526), 35335)


...

('jaccard', 50, 0.2, False, np.float64(0.9997585467334216), 123975)


('jaccard', 100, 0, False, np.float64(0.9782261085995155), 5241)


('jaccard', 100, 0.1, False, np.float64(0.9827512192692165), 26956)




Similarity metric: 100%|██████████| 3/3 [08:57<00:00, 179.12s/it]

('jaccard', 100, 0.2, False, np.float64(1.0008711005646924), 123880)


In [27]:
results

[('cosine', 25, 0, False, np.float64(1.0031778877693514), 13961),
 ('cosine', 25, 0.1, False, np.float64(1.0029982138508473), 14284),
 ('cosine', 25, 0.2, False, np.float64(1.0032462710724526), 35335),
 ('cosine', 50, 0, False, np.float64(0.9913165686950676), 7832),
 ('cosine', 50, 0.1, False, np.float64(0.9911853270521767), 8435),
 ('cosine', 50, 0.2, False, np.float64(0.9941334649525436), 32316),
 ('cosine', 100, 0, False, np.float64(0.9891763670354068), 4369),
 ('cosine', 100, 0.1, False, np.float64(0.9893182296772677), 5306),
 ('cosine', 100, 0.2, False, np.float64(0.9929645524763733), 31214),
 ('pearson', 25, 0, False, np.float64(1.0032048512410878), 13970),
 ('pearson', 25, 0.1, False, np.float64(1.003021597399783), 14291),
 ('pearson', 25, 0.2, False, np.float64(1.0031172911012074), 35383),
 ('pearson', 50, 0, False, np.float64(0.9914112606246815), 7832),
 ('pearson', 50, 0.1, False, np.float64(0.991282106977537), 8437),
 ('pearson', 50, 0.2, False, np.float64(0.9941092723963285

## Implementación de ponderación de McLaughlin

Se calculará McNally para el mejor tipo de similitud en el modelo item-item, que en este caso es cosine, para esta se iterará para 3 Valores de k vecinos, 3 valores de umbral de similitud, y 5 posibles valores de ponderación de McLaughlin (incluyendo NO implementarla)

## Submuestra de test (n=5.000)

In [28]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

In [29]:
for metric in tqdm(["cosine"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/items_{metric}_top100_co_rated.npz") as data:
        ttop_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.1, 0.2], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse, missing = evaluate_rmse_item(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse, missing))
                    print((metric, K, sim_threshold, weight, None, rmse, missing))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse, missing = evaluate_rmse_item(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse, missing))
                        print((metric, K, sim_threshold, weight, beta, rmse, missing))

    

Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0
min co_rated: 0
max co_rated: 3642


('cosine', 25, 0, False, None, np.float64(1.0056557408107982), 269)
('cosine', 25, 0, True, 5, np.float64(1.0126385777047253), 269)
('cosine', 25, 0, True, 10, np.float64(1.0157086927550556), 269)
('cosine', 25, 0, True, 25, np.float64(1.0193661703168475), 269)
('cosine', 25, 0, True, 50, np.float64(1.0214319011596535), 269)


('cosine', 25, 0, True, 100, np.float64(1.0221340048526006), 269)
('cosine', 25, 0.1, False, None, np.float64(1.0040764327788705), 282)
('cosine', 25, 0.1, True, 5, np.float64(1.0111102285172748), 282)
('cosine', 25, 0.1, True, 10, np.float64(1.0141990753937293), 282)
('cosine', 25, 0.1, True, 25, np.float64(1.0179348090883198), 282)
('cosine', 25, 0.1, True, 50, np.float64(1.0200200156727386), 282)


('cosine', 25, 0.1, True, 100, np.float64(1.0207415294189264), 282)
('cosine', 25, 0.2, False, None, np.float64(1.0174789935620738), 693)
('cosine', 25, 0.2, True, 5, np.float64(1.0234255562743513), 693)
('cosine', 25, 0.2, True, 10, np.float64(1.0258050390061981), 693)
('cosine', 25, 0.2, True, 25, np.float64(1.0291562936276364), 693)
('cosine', 25, 0.2, True, 50, np.float64(1.0309780771200197), 693)


...

('cosine', 100, 0, False, None, np.float64(0.9819672028860388), 76)
('cosine', 100, 0, True, 5, np.float64(0.9850647099463227), 76)
('cosine', 100, 0, True, 10, np.float64(0.9875792172572956), 76)
('cosine', 100, 0, True, 25, np.float64(0.9901409535083007), 76)
('cosine', 100, 0, True, 50, np.float64(0.9919377749551777), 76)


('cosine', 100, 0, True, 100, np.float64(0.9928444176866846), 76)
('cosine', 100, 0.1, False, None, np.float64(0.9825753650531198), 96)
('cosine', 100, 0.1, True, 5, np.float64(0.9853379050875283), 96)
('cosine', 100, 0.1, True, 10, np.float64(0.9877181028872085), 96)
('cosine', 100, 0.1, True, 25, np.float64(0.9901373759601595), 96)
('cosine', 100, 0.1, True, 50, np.float64(0.9920182113889486), 96)


('cosine', 100, 0.1, True, 100, np.float64(0.9930036490384511), 96)
('cosine', 100, 0.2, False, None, np.float64(0.9996290935273029), 599)
('cosine', 100, 0.2, True, 5, np.float64(1.0039109071557024), 599)
('cosine', 100, 0.2, True, 10, np.float64(1.006778749216797), 599)
('cosine', 100, 0.2, True, 25, np.float64(1.009061437197997), 599)
('cosine', 100, 0.2, True, 50, np.float64(1.0102070182494762), 599)




Similarity metric: 100%|██████████| 1/1 [00:21<00:00, 21.26s/it]

('cosine', 100, 0.2, True, 100, np.float64(1.0107327930583594), 599)


In [30]:
results

[('cosine', 25, 0, False, None, np.float64(1.0056557408107982), 269),
 ('cosine', 25, 0, True, 5, np.float64(1.0126385777047253), 269),
 ('cosine', 25, 0, True, 10, np.float64(1.0157086927550556), 269),
 ('cosine', 25, 0, True, 25, np.float64(1.0193661703168475), 269),
 ('cosine', 25, 0, True, 50, np.float64(1.0214319011596535), 269),
 ('cosine', 25, 0, True, 100, np.float64(1.0221340048526006), 269),
 ('cosine', 25, 0.1, False, None, np.float64(1.0040764327788705), 282),
 ('cosine', 25, 0.1, True, 5, np.float64(1.0111102285172748), 282),
 ('cosine', 25, 0.1, True, 10, np.float64(1.0141990753937293), 282),
 ('cosine', 25, 0.1, True, 25, np.float64(1.0179348090883198), 282),
 ('cosine', 25, 0.1, True, 50, np.float64(1.0200200156727386), 282),
 ('cosine', 25, 0.1, True, 100, np.float64(1.0207415294189264), 282),
 ('cosine', 25, 0.2, False, None, np.float64(1.0174789935620738), 693),
 ('cosine', 25, 0.2, True, 5, np.float64(1.0234255562743513), 693),
 ('cosine', 25, 0.2, True, 10, np.floa

## Submuestra de test (n=250.000)

In [31]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(250_000, random_state=42)

In [32]:
for metric in tqdm(["cosine"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())
    with np.load(f"Train_neighbours/items_{metric}_top100_co_rated.npz") as data:
        ttop_k_co_rated = data["co_rated"]
    print("min co_rated:", top_k_co_rated.min())
    print("max co_rated:", top_k_co_rated.max())

    for K in tqdm([25,50,100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]
        co_rated = top_k_co_rated[:, :K]

        for sim_threshold in tqdm([0, 0.1, 0.2], desc="sim_threshold", leave=False):
            # comparar sin y con ponderación de McLaughlin
            for weight in [False, True]:
                if not weight:
                    rmse, missing = evaluate_rmse_item(
                        R_train,
                        test_filtered,
                        neighbours,
                        sims,
                        K,
                        sim_threshold,
                        means,
                        weighting=False,
                        B=None
                    )

                    results.append((metric, K, sim_threshold, weight, None, rmse, missing))
                    print((metric, K, sim_threshold, weight, None, rmse, missing))
                else:
                    for beta in [5, 10, 25, 50, 100]:
                        rmse, missing = evaluate_rmse_item(
                            R_train,
                            test_filtered,
                            neighbours,
                            sims,
                            K,
                            sim_threshold,
                            means,
                            weighting=True,
                            B=co_rated,
                            beta=beta
                        )

                        results.append((metric, K, sim_threshold, weight, beta, rmse, missing))
                        print((metric, K, sim_threshold, weight, beta, rmse, missing))

    

Similarity metric:   0%|          | 0/1 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0
min co_rated: 0
max co_rated: 3642


('cosine', 25, 0, False, None, np.float64(1.0031778877693514), 13961)
('cosine', 25, 0, True, 5, np.float64(1.0072107972053828), 13973)
('cosine', 25, 0, True, 10, np.float64(1.0102499598554469), 13973)
('cosine', 25, 0, True, 25, np.float64(1.0141272466170776), 13973)
('cosine', 25, 0, True, 50, np.float64(1.016337502623393), 13973)


('cosine', 25, 0, True, 100, np.float64(1.0175397008774734), 13973)
('cosine', 25, 0.1, False, None, np.float64(1.0029982138508473), 14284)
('cosine', 25, 0.1, True, 5, np.float64(1.0070161365305288), 14296)
('cosine', 25, 0.1, True, 10, np.float64(1.010043608901122), 14296)
('cosine', 25, 0.1, True, 25, np.float64(1.0139083831916802), 14296)
('cosine', 25, 0.1, True, 50, np.float64(1.0161188698805201), 14296)


('cosine', 25, 0.1, True, 100, np.float64(1.0173250669616511), 14296)
('cosine', 25, 0.2, False, None, np.float64(1.0032462710724526), 35335)
('cosine', 25, 0.2, True, 5, np.float64(1.0070468808838804), 35346)
('cosine', 25, 0.2, True, 10, np.float64(1.0097392408732384), 35346)
('cosine', 25, 0.2, True, 25, np.float64(1.01342685544154), 35346)
('cosine', 25, 0.2, True, 50, np.float64(1.015578547235932), 35346)


...

('cosine', 100, 0, False, None, np.float64(0.9891763670354068), 4369)
('cosine', 100, 0, True, 5, np.float64(0.9911203842963496), 4383)
('cosine', 100, 0, True, 10, np.float64(0.9927350429300845), 4383)
('cosine', 100, 0, True, 25, np.float64(0.9952292406919283), 4383)
('cosine', 100, 0, True, 50, np.float64(0.9966000409085767), 4383)


('cosine', 100, 0, True, 100, np.float64(0.9972327514672497), 4383)
('cosine', 100, 0.1, False, None, np.float64(0.9893182296772677), 5306)
('cosine', 100, 0.1, True, 5, np.float64(0.9912782025317421), 5320)
('cosine', 100, 0.1, True, 10, np.float64(0.9929009908113083), 5320)
('cosine', 100, 0.1, True, 25, np.float64(0.9954011603816446), 5320)
('cosine', 100, 0.1, True, 50, np.float64(0.9967824957612206), 5320)


('cosine', 100, 0.1, True, 100, np.float64(0.9974193653980424), 5320)
('cosine', 100, 0.2, False, None, np.float64(0.9929645524763733), 31214)
('cosine', 100, 0.2, True, 5, np.float64(0.9951059100934941), 31225)
('cosine', 100, 0.2, True, 10, np.float64(0.9967153417971407), 31225)
('cosine', 100, 0.2, True, 25, np.float64(0.999367456210805), 31225)
('cosine', 100, 0.2, True, 50, np.float64(1.0008051714384263), 31225)




Similarity metric: 100%|██████████| 1/1 [16:56<00:00, 1016.62s/it]

('cosine', 100, 0.2, True, 100, np.float64(1.0014259637622873), 31225)


In [33]:
results

[('cosine', 25, 0, False, None, np.float64(1.0031778877693514), 13961),
 ('cosine', 25, 0, True, 5, np.float64(1.0072107972053828), 13973),
 ('cosine', 25, 0, True, 10, np.float64(1.0102499598554469), 13973),
 ('cosine', 25, 0, True, 25, np.float64(1.0141272466170776), 13973),
 ('cosine', 25, 0, True, 50, np.float64(1.016337502623393), 13973),
 ('cosine', 25, 0, True, 100, np.float64(1.0175397008774734), 13973),
 ('cosine', 25, 0.1, False, None, np.float64(1.0029982138508473), 14284),
 ('cosine', 25, 0.1, True, 5, np.float64(1.0070161365305288), 14296),
 ('cosine', 25, 0.1, True, 10, np.float64(1.010043608901122), 14296),
 ('cosine', 25, 0.1, True, 25, np.float64(1.0139083831916802), 14296),
 ('cosine', 25, 0.1, True, 50, np.float64(1.0161188698805201), 14296),
 ('cosine', 25, 0.1, True, 100, np.float64(1.0173250669616511), 14296),
 ('cosine', 25, 0.2, False, None, np.float64(1.0032462710724526), 35335),
 ('cosine', 25, 0.2, True, 5, np.float64(1.0070468808838804), 35346),
 ('cosine', 